# Amazn AI - Day 4
## Connecting Gemini LLM with RAG

## Step 1: Load API Key

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
print("API Loaded" if GOOGLE_API_KEY else "Missing API Key")

## Step 2: Initialize Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

## Step 3: Prompt

In [ ]:
prompt=ChatPromptTemplate.from_template("""
You are Amazn AI.
Use ONLY the provided context.

Context:
{context}

Question:
{question}

Answer:
""")

## Step 4: RAG Function

In [ ]:
def rag_with_llm(query_text,vector_store,model,top_k=3,**filters):
    query=model.encode([query_text])[0]
    docs=vector_store.similarity_search_by_vector(query,k=top_k*3)
    filtered=[]
    for d in docs:
        m=d.metadata
        if filters.get("category") and filters["category"].lower() not in m.get("category","").lower(): continue
        if filters.get("min_price") is not None and m.get("discounted_price",0)<filters["min_price"]: continue
        if filters.get("max_price") is not None and m.get("discounted_price",1e9)>filters["max_price"]: continue
        if filters.get("rating") is not None and m.get("rating",0)<filters["rating"]: continue
        if filters.get("product_name") and filters["product_name"].lower() not in m.get("product_name","").lower(): continue
        filtered.append(d)
        if len(filtered)>=top_k: break
    context="\n\n".join(doc.page_content for doc in filtered)
    chain=prompt|llm
    response=chain.invoke({"context":context,"question":query_text})
    return response.content,filtered

## Step 5: Test

In [ ]:
query='best gaming laptop under 50000'
answer,docs=rag_with_llm(query,vector_store,model,top_k=3)
print(answer)
for d in docs:
    print(f"{d.metadata['product_name']} : {d.metadata.get('product_link','')}")

## Day 4 Complete
Connected Gemini to the retrieval pipeline. Day 5 will focus on production improvements.